# MuseTalk 1.5 production lip-sync stage (private Kaggle GPU)

This notebook supports one or more approved presenter/audio pairs in a single GPU session. The private dataset includes a `musetalk_manifest.json` describing each pair. MuseTalk 1.5 loads its model once, then renders the requested presenter tasks to separate MP4 files.

**Natural-face policy:** use the tested MuseTalk 1.5 path, preserve the source face proportions and the original presenter footage, and do not post-stretch the mouth or lips. The upstream MuseTalk 1.5 inference path fixes `bbox_shift` at 0; that parameter is not used for v1.5. This notebook uses the default v1.5 face parser with `extra_margin=10` and `parsing_mode=jaw`, matching the validated preview pipeline.

**Budget: USD 0.** The workflow requests Kaggle's free NVIDIA T4 GPU pool. If no free GPU is available, stop rather than use a paid GPU or unsynchronized presenter fallback.

The notebook can still run in legacy single-video / single-audio mode for a short private review, but production submissions should use the manifest format.


In [ ]:
import glob, os, shutil, subprocess, torch

print("Kaggle base-image PyTorch:", torch.__version__)
print("Base-image CUDA visibility:", torch.cuda.is_available())

# Some Kaggle base images omit the nvidia-smi binary. Check device nodes instead,
# then validate real CUDA access after installing the dedicated MuseTalk CUDA wheel.
gpu_nodes = sorted(glob.glob("/dev/nvidia*"))
print("NVIDIA device nodes:", gpu_nodes)
if not any(os.path.basename(p).startswith("nvidia") and p.endswith(("0", "ctl")) for p in gpu_nodes):
    raise RuntimeError(
        "Kaggle accepted the GPU notebook metadata but did not mount an NVIDIA device. "
        "Free GPU quota/capacity may be unavailable. Stop here; do not use CPU or paid fallback."
    )

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    probe = subprocess.run([nvidia_smi, "-L"], capture_output=True, text=True)
    print("nvidia-smi:", probe.stdout.strip() or probe.stderr.strip())
else:
    print("nvidia-smi executable is absent; the CUDA-enabled MuseTalk venv will verify the GPU in the install cell.")


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq ffmpeg
git clone https://github.com/TMElyralab/MuseTalk.git /kaggle/working/MuseTalk
cd /kaggle/working/MuseTalk

# Upstream MuseTalk pins NumPy/TensorFlow to versions that do not support
# Kaggle's current Python 3.13 image. Use the project's recommended Python 3.10.
python -m pip install -q --upgrade uv
uv python install 3.10
# Seed the environment with pip/setuptools/wheel. uv otherwise creates a
# pip-free environment, so "python -m pip" would fail on the next line.
uv venv --seed /kaggle/working/musetalk-venv --python 3.10
MUSEPY=/kaggle/working/musetalk-venv/bin/python
MUSEBIN=/kaggle/working/musetalk-venv/bin
"$MUSEPY" -m pip --version
"$MUSEPY" -m pip install -q --upgrade pip setuptools wheel

# MuseTalk's documented CUDA 11.8 build supports Kaggle's NVIDIA T4 pool.
"$MUSEPY" -m pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 \
  --index-url https://download.pytorch.org/whl/cu118
"$MUSEPY" -m pip install -q -r requirements.txt
# The upstream MuseTalk checkout contains source files under musetalk/whisper,
# but that folder has no setup.py or pyproject.toml. Inference uses the
# Transformers WhisperModel and does not require pip-installing that folder.

# Install the OpenMMLab runtime components required by MuseTalk's DWPose
# preprocessing. These match the upstream MuseTalk installation guidance.
"$MUSEPY" -m pip install -q --no-cache-dir -U openmim
"$MUSEBIN/mim" install -q mmengine
"$MUSEBIN/mim" install -q "mmcv==2.0.1"
"$MUSEBIN/mim" install -q "mmdet==3.1.0"
# MMPose declares chumpy (legacy SMPL fitting) as a setup dependency; MuseTalk DWPose inference does not use it.
# Install MMPose without optional/legacy deps to avoid chumpy failing to build on modern pip.
"$MUSEPY" -m pip install -q --no-deps "mmpose==1.1.0"
"$MUSEPY" -m pip install -q json_tricks munkres xtcocotools yapf
"$MUSEPY" -m pip install -q huggingface_hub==0.30.2 gdown

# MuseTalk's preprocessing imports its bundled face_detection package by
# its top-level module name; expose that source folder as well as the repo root.
export PYTHONPATH="$PWD/musetalk/utils:$PWD${PYTHONPATH:+:$PYTHONPATH}"
# Kaggle sets an IPython-only Matplotlib backend. MuseTalk runs in a separate
# Python 3.10 environment, so force a headless backend before importing MMPose.
export MPLBACKEND=Agg
export MPLCONFIGDIR=/kaggle/working/matplotlib-config
mkdir -p "$MPLCONFIGDIR"

# Fail here with a clear message instead of later during video inference.
"$MUSEPY" - <<'PY'
import torch, matplotlib
if matplotlib.get_backend().lower() != "agg":
    raise RuntimeError(f"Expected Matplotlib Agg backend, got {matplotlib.get_backend()}")
print("Matplotlib backend:", matplotlib.get_backend())
import mmcv, mmdet, mmpose
from transformers import WhisperModel
from musetalk.utils.audio_processor import AudioProcessor
from face_detection import FaceAlignment, LandmarksType
from mmpose.apis import inference_topdown, init_model

print("MuseTalk PyTorch:", torch.__version__)
print("MuseTalk CUDA:", torch.cuda.is_available())
print("MuseTalk GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("mmcv:", mmcv.__version__)
print("mmdet:", mmdet.__version__)
print("mmpose:", mmpose.__version__)
print("WhisperModel, AudioProcessor and face_detection imports: OK")
if not torch.cuda.is_available():
    raise RuntimeError("MuseTalk Python 3.10 environment cannot see the free Kaggle GPU.")
PY

# Prove the dedicated Python environment can see the requested GPU.
"$MUSEPY" - <<'PY'
import torch
print("MuseTalk environment PyTorch:", torch.__version__)
print("MuseTalk environment CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("MuseTalk Python 3.10 environment cannot see the free Kaggle GPU.")
print("MuseTalk GPU:", torch.cuda.get_device_name(0))
PY

# The upstream downloader otherwise upgrades huggingface_hub without a pin.
python - <<'PY'
from pathlib import Path
p = Path("download_weights.sh")
s = p.read_text()
s = s.replace(
    'pip install -U "huggingface_hub[cli]"',
    'pip install "huggingface_hub[cli]==0.30.2"'
)
p.write_text(s)
PY

echo "MuseTalk source and Python 3.10 CUDA dependencies installed."


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/MuseTalk

# Replace the upstream non-fail-fast model download script. Every required
# V1.5 asset is downloaded directly, retried, size-checked and hash-checked.

download_verified() {
  local dest="$1" min_bytes="$2" expected_sha="$3"
  shift 3
  local tmp="$dest.part" url bytes actual
  mkdir -p "$(dirname "$dest")"

  if [[ -s "$dest" ]]; then
    bytes="$(stat -c%s "$dest")"
    if [[ "$bytes" -ge "$min_bytes" ]]; then
      if [[ "$expected_sha" == "-" ]] || [[ "$(sha256sum "$dest" | awk '{print $1}')" == "$expected_sha" ]]; then
        echo "Verified cached asset: $dest ($bytes bytes)"
        return 0
      fi
    fi
    rm -f "$dest"
  fi

  for url in "$@"; do
    echo "Downloading $dest from $url"
    rm -f "$tmp"
    if curl --fail --location --show-error --progress-bar \
      --retry 4 --retry-delay 3 --retry-all-errors \
      --connect-timeout 30 --max-time 7200 \
      --output "$tmp" "$url"; then
      bytes="$(stat -c%s "$tmp")"
      if [[ "$bytes" -lt "$min_bytes" ]]; then
        echo "Rejected $url: $bytes bytes; expected at least $min_bytes." >&2
        rm -f "$tmp"
        continue
      fi
      if [[ "$expected_sha" != "-" ]]; then
        actual="$(sha256sum "$tmp" | awk '{print $1}')"
        if [[ "$actual" != "$expected_sha" ]]; then
          echo "Rejected $url: SHA256 mismatch ($actual)." >&2
          rm -f "$tmp"
          continue
        fi
      fi
      mv "$tmp" "$dest"
      echo "Verified $dest ($bytes bytes)"
      return 0
    fi
    echo "Download source failed: $url" >&2
  done

  echo "::error::Could not download and verify required model asset: $dest" >&2
  return 1
}

# MuseTalk V1.5 UNet (primary and duplicate mirrors share this SHA256).
download_verified models/musetalkV15/unet.pth 3400000000 \
  7ebf6c98c181e20838e4c0054e96e944ac60d5d692cc01db42839fe11b787007 \
  "https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth?download=true" \
  "https://huggingface.co/skye1128/MuseTalk/resolve/main/musetalkV15/unet.pth?download=true" \
  "https://hf-mirror.com/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth"

download_verified models/musetalkV15/musetalk.json 700 - \
  "https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json?download=true" \
  "https://huggingface.co/skye1128/MuseTalk/resolve/main/musetalkV15/musetalk.json?download=true" \
  "https://hf-mirror.com/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json"

# Stable Diffusion VAE (model loader expects models/sd-vae).
download_verified models/sd-vae/config.json 500 - \
  "https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/config.json?download=true" \
  "https://hf-mirror.com/stabilityai/sd-vae-ft-mse/resolve/main/config.json"
download_verified models/sd-vae/diffusion_pytorch_model.bin 300000000 \
  1b4889b6b1d4ce7ae320a02dedaeff1780ad77d415ea0d744b476155c6377ddc \
  "https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin?download=true" \
  "https://huggingface.co/diffusers/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin?download=true" \
  "https://hf-mirror.com/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin"

# Whisper tiny encoder.
download_verified models/whisper/config.json 500 - \
  "https://huggingface.co/openai/whisper-tiny/resolve/main/config.json?download=true" \
  "https://hf-mirror.com/openai/whisper-tiny/resolve/main/config.json"
download_verified models/whisper/preprocessor_config.json 100 - \
  "https://huggingface.co/openai/whisper-tiny/resolve/main/preprocessor_config.json?download=true" \
  "https://hf-mirror.com/openai/whisper-tiny/resolve/main/preprocessor_config.json"
download_verified models/whisper/pytorch_model.bin 140000000 \
  9607f98a2b22d9e229ae43c52ecea79dcede9e0c5cfae67e8da6eda86d8aac1d \
  "https://huggingface.co/openai/whisper-tiny/resolve/main/pytorch_model.bin?download=true" \
  "https://hf-mirror.com/openai/whisper-tiny/resolve/main/pytorch_model.bin"

# DWPose; fallback mirrors resolve to the same verified 407 MB checkpoint.
download_verified models/dwpose/dw-ll_ucoco_384.pth 400000000 \
  0d9408b13cd863c4e95a149dd31232f88f2a12aa6cf8964ed74d7d97748c7a07 \
  "https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth?download=true" \
  "https://huggingface.co/camenduru/DWPose/resolve/main/dw-ll_ucoco_384.pth?download=true" \
  "https://huggingface.co/camenduru/MuseTalk/resolve/5cfe77e50e1dc10ac599e5340d610d69a5086af2/dwpose/dw-ll_ucoco_384.pth?download=true" \
  "https://hf-mirror.com/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth"

# Face parser and ResNet18 backbone.
download_verified models/face-parse-bisent/79999_iter.pth 50000000 \
  468e13ca13a9b43cc0881a9f99083a430e9c0a38abd935431d1c28ee94b26567 \
  "https://huggingface.co/ManyOtherFunctions/face-parse-bisent/resolve/9b440738b337fd51492d718685e452af3532acd0/79999_iter.pth?download=true" \
  "https://hf-mirror.com/ManyOtherFunctions/face-parse-bisent/resolve/main/79999_iter.pth"
download_verified models/face-parse-bisent/resnet18-5c106cde.pth 40000000 - \
  "https://download.pytorch.org/models/resnet18-5c106cde.pth"

# Preload SFD detector. Upstream otherwise downloads it silently during inference.
download_verified musetalk/utils/face_detection/detection/sfd/s3fd.pth 80000000 \
  619a31681264d3f7f7fc7a16a42cbbe8b23f31a256f75a366e5a1bcd59b33543 \
  "https://huggingface.co/ByteDance/LatentSync/resolve/main/auxiliary/s3fd-619a316812.pth?download=true" \
  "https://huggingface.co/wsj1995/sadTalker/resolve/main/s3fd-619a316812.pth?download=true" \
  "https://hf-mirror.com/ByteDance/LatentSync/resolve/main/auxiliary/s3fd-619a316812.pth" \
  "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth"

# Do not let inference run with incomplete or HTML/error-page model files.
python - <<'PY'
import json
from pathlib import Path

required = {
    "models/musetalkV15/unet.pth": 3_400_000_000,
    "models/musetalkV15/musetalk.json": 700,
    "models/sd-vae/config.json": 500,
    "models/sd-vae/diffusion_pytorch_model.bin": 300_000_000,
    "models/whisper/config.json": 500,
    "models/whisper/preprocessor_config.json": 100,
    "models/whisper/pytorch_model.bin": 140_000_000,
    "models/dwpose/dw-ll_ucoco_384.pth": 400_000_000,
    "models/face-parse-bisent/79999_iter.pth": 50_000_000,
    "models/face-parse-bisent/resnet18-5c106cde.pth": 40_000_000,
    "musetalk/utils/face_detection/detection/sfd/s3fd.pth": 80_000_000,
}
for name, minimum in required.items():
    p = Path(name)
    if not p.is_file() or p.stat().st_size < minimum:
        raise SystemExit(f"Required model asset missing/truncated: {name}")
    print(f"PRECHECK OK: {name} ({p.stat().st_size:,} bytes)")

for name in (
    "models/musetalkV15/musetalk.json",
    "models/sd-vae/config.json",
    "models/whisper/config.json",
    "models/whisper/preprocessor_config.json",
):
    with open(name, encoding="utf-8") as f:
        json.load(f)
print("All required V1.5 model assets passed size, checksum and JSON checks.")
PY


In [ ]:
# Stage one or more presenter/audio pairs from the attached private Kaggle Dataset.
from pathlib import Path
import os, glob, shutil, json, re

input_root = Path('/kaggle/input')
manifest_candidates = sorted(input_root.rglob('musetalk_manifest.json'))
if len(manifest_candidates) > 1:
    raise RuntimeError('Attach only one dataset with a musetalk_manifest.json file.')

work = Path('/kaggle/working/MuseTalk')
data_dir = work / 'data' / 'free_test'
data_dir.mkdir(parents=True, exist_ok=True)
tasks = []
return_config = None

def stage_pair(index, video_source, audio_source, result_name):
    video_source = Path(video_source).resolve()
    audio_source = Path(audio_source).resolve()
    if not video_source.is_relative_to(input_root.resolve()) or not video_source.is_file():
        raise RuntimeError(f'Presenter video is missing or outside Kaggle inputs: {video_source}')
    if not audio_source.is_relative_to(input_root.resolve()) or not audio_source.is_file():
        raise RuntimeError(f'Narration audio is missing or outside Kaggle inputs: {audio_source}')
    if not re.fullmatch(r'(?:musetalk_preview|musetalk_preview_[A-Za-z0-9_-]+)[.]mp4', result_name):
        raise RuntimeError(f'Invalid MuseTalk output filename: {result_name}')
    video_target = data_dir / f'presenter_{index}.mp4'
    audio_target = data_dir / f'narration_{index}{audio_source.suffix.lower()}'
    shutil.copy2(video_source, video_target)
    shutil.copy2(audio_source, audio_target)
    tasks.append({
        'id': f'task_{index}',
        'video_path': str(video_target),
        'audio_path': str(audio_target),
        'result_name': result_name,
    })

if manifest_candidates:
    manifest_path = manifest_candidates[0]
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    return_config = manifest.get('return')
    if return_config is not None:
        if not isinstance(return_config, dict):
            raise RuntimeError('MuseTalk return configuration must be an object.')
        base_url = str(return_config.get('base_url') or '')
        route = str(return_config.get('route') or '')
        token = str(return_config.get('token') or '')
        if not base_url.startswith('https://') or not re.fullmatch(r'[a-f0-9]{48}', route) or not re.fullmatch(r'[a-f0-9]{64}', token):
            raise RuntimeError('MuseTalk private return relay configuration is invalid.')
        print('A temporary authenticated output relay is configured for this private batch.')
    task_specs = manifest.get('tasks', [])
    if not isinstance(task_specs, list) or not task_specs:
        raise RuntimeError('musetalk_manifest.json must contain a non-empty tasks array.')
    if len(task_specs) > 4:
        raise RuntimeError(f'At most four presenter looks are supported per kernel; got {len(task_specs)}.')
    for index, spec in enumerate(task_specs):
        if not isinstance(spec, dict):
            raise RuntimeError(f'Task {index} must be an object.')
        # Dataset assets live beside the manifest under /kaggle/input/<dataset>/.
        # Resolve relative to that exact dataset folder, not the /kaggle/input root.
        stage_pair(
            index,
            manifest_path.parent / spec['video'],
            manifest_path.parent / spec['audio'],
            str(spec.get('result_name') or f'musetalk_preview_{index}.mp4'),
        )
else:
    # Backward-compatible single-pair mode for the original private preview.
    video_candidates = [p for p in input_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.mp4','.mov','.mkv'}]
    audio_candidates = [p for p in input_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.wav','.mp3','.m4a'}]
    video_candidates = [p for p in video_candidates if p.name != 'musetalk_preview.mp4']
    audio_candidates = [p for p in audio_candidates if p.name not in {'musetalk_manifest.json'}]
    if len(video_candidates) != 1 or len(audio_candidates) != 1:
        raise RuntimeError(
            'Legacy mode requires exactly one presenter video and one narration audio file; '
            'for multiple pairs, attach a musetalk_manifest.json.'
        )
    stage_pair(0, video_candidates[0], audio_candidates[0], 'musetalk_preview.mp4')

print(f'Staged {len(tasks)} presenter/audio pair(s):')
for task in tasks:
    print({key: task[key] for key in ('id','video_path','audio_path','result_name')})


In [ ]:
# Configure the requested presenter/audio pairs for a single MuseTalk 1.5 model session.
import yaml
config = {}
for task in tasks:
    config[task['id']] = {
        'video_path': task['video_path'],
        'audio_path': task['audio_path'],
        'result_name': task['result_name'],
    }

config_path = work / 'configs' / 'inference' / 'kaggle_free_test.yaml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)
print(f'Inference config has {len(config)} task(s):')
print(config_path.read_text())


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/MuseTalk
STREAM_PATCH="$(find /kaggle/input -name 'musetalk_streaming_patch.py' -print -quit)"
if [ -z "$STREAM_PATCH" ]; then
  echo "::error::Missing private MuseTalk streaming patch from the input dataset."
  exit 1
fi
/kaggle/working/musetalk-venv/bin/python "$STREAM_PATCH" /kaggle/working/MuseTalk/scripts/inference.py
echo "Kaggle working-disk capacity before inference:"
df -h /kaggle/working
du -sh /kaggle/working/MuseTalk/models /kaggle/working/MuseTalk/data 2>/dev/null || true
export PYTHONPATH="$PWD/musetalk/utils:$PWD${PYTHONPATH:+:$PYTHONPATH}"
export MPLBACKEND=Agg
export MPLCONFIGDIR=/kaggle/working/matplotlib-config
mkdir -p "$MPLCONFIGDIR"
/kaggle/working/musetalk-venv/bin/python -m scripts.inference \
  --inference_config configs/inference/kaggle_free_test.yaml \
  --result_dir /kaggle/working/musetalk_preview \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json \
  --version v15 \
  --fps 25 \
  --batch_size 4 \
  --use_float16 \
  --extra_margin 10 \
  --parsing_mode jaw \
  --left_cheek_width 90 \
  --right_cheek_width 90


In [ ]:
# Export and validate every completed presenter render.
from pathlib import Path
from IPython.display import Video, display
import hashlib
import json
import re
import shutil
import subprocess
import time
import requests
from fractions import Fraction

results_dir = Path('/kaggle/working/musetalk_preview/v15')
export_dir = Path('/kaggle/working')
summaries = []

for task in tasks:
    rendered = results_dir / task['result_name']
    if not rendered.is_file():
        candidates = sorted(str(p) for p in results_dir.rglob('*.mp4'))
        raise RuntimeError(
            f'MuseTalk did not create {task["result_name"]}. '
            f'Available MP4 candidates: {candidates[:20]}. No unsynchronized fallback will be used.'
        )

    probe = json.loads(subprocess.check_output([
        'ffprobe', '-v', 'error', '-show_streams', '-show_format', '-of', 'json', str(rendered)
    ], text=True))
    streams = probe.get('streams', [])
    stream_types = {stream.get('codec_type') for stream in streams}
    video_stream = next((s for s in streams if s.get('codec_type') == 'video'), {})
    frame_rate = Fraction(video_stream.get('avg_frame_rate') or video_stream.get('r_frame_rate') or '0/1')
    duration = float(probe.get('format', {}).get('duration', 0))
    size = rendered.stat().st_size
    if frame_rate != Fraction(25, 1):
        raise RuntimeError(f'MuseTalk output must preserve preview cadence of 25 fps; got {frame_rate}')
    if size < 100_000 or duration < 2:
        raise RuntimeError(f'Invalid MuseTalk output size/duration: {size} bytes, {duration:.2f}s')
    if not {'video', 'audio'}.issubset(stream_types):
        raise RuntimeError(f'MuseTalk output is missing audio or video: {sorted(stream_types)}')

    exported = export_dir / task['result_name']
    shutil.copy2(rendered, exported)
    summaries.append({
        'task_id': task['id'],
        'file': str(exported),
        'bytes': exported.stat().st_size,
        'duration_seconds': duration,
        'fps': float(frame_rate),
        'streams': sorted(stream_types),
    })
    print(json.dumps(summaries[-1], indent=2))

    # Return the rendered MP4 directly to this one GitHub Actions runner.
    # The callback is per-run, bearer-protected, filename allowlisted and expires
    # when cloudflared is terminated by the waiting runner.
    if return_config:
        filename = task['result_name']
        digest = hashlib.sha256(exported.read_bytes()).hexdigest()
        relay_url = (
            f"{return_config['base_url'].rstrip('/')}"
            f"/upload/{return_config['route']}/{filename}"
        )
        headers = {
            'Authorization': f"Bearer {return_config['token']}",
            'Content-Type': 'video/mp4',
            'Content-Length': str(exported.stat().st_size),
            'X-Content-SHA256': digest,
        }
        last_error = None
        for attempt in range(1, 4):
            try:
                with exported.open('rb') as video_file:
                    response = requests.put(
                        relay_url,
                        headers=headers,
                        data=video_file,
                        timeout=(20, 600),
                    )
                if response.status_code == 201:
                    confirmation = response.json()
                    if confirmation.get('ok') is not True or confirmation.get('file') != filename or confirmation.get('sha256') != digest:
                        raise RuntimeError('Private output relay returned an invalid upload receipt.')
                    print(f"Private return relay accepted {filename} ({exported.stat().st_size:,} bytes).")
                    last_error = None
                    break
                last_error = RuntimeError(f"Private relay returned HTTP {response.status_code}.")
            except Exception as exc:
                last_error = exc
            if attempt < 3:
                time.sleep(3 * attempt)
        if last_error is not None:
            print(
                f"Private return relay could not transfer {filename}: {str(last_error)[:240]}. "
                "The runner will try Kaggle's normal output download.",
                flush=True,
            )

    display(Video(str(exported), embed=True))

summary_path = export_dir / 'musetalk_preview_manifest.json'
summary_path.write_text(json.dumps({'renders': summaries}, indent=2), encoding='utf-8')
print(f'Validated {len(summaries)} MuseTalk 1.5 render(s): {summary_path}')
